# 30a — Variational Autoencoders

Corresponds to: Note 30 — Variational Autoencoders

**Libraries introduced in this notebook:** No new libraries — PyTorch (`torch`, `torch.nn`) was introduced in Note 09.

**What this notebook demonstrates:**
1. Encoder forward pass trace matching the Note 30 worked example — computing $\mu$, $\log\sigma^2$, and the reparameterised sample $z$
2. Closed-form KL divergence computation and verification against the worked example
3. Full VAE implementation on synthetic data — encoder, reparameterisation, decoder, ELBO loss
4. Generation: sampling $z \sim \mathcal{N}(0, I)$ and decoding to produce new outputs

## Part 1 — Encoder Forward Pass Trace

We reproduce the Note 30 worked example exactly: a VAE encoder with $d=3$, $k=2$, computing $\mu$, $\log\sigma^2$, $\sigma$, and the reparameterised sample $z$.

In [1]:
import torch
import torch.nn as nn

# ── Encoder weights (from Note 30) ────────────────────────────────────────────
W_mu = torch.tensor([[ 0.4, -0.2,  0.6],
                     [-0.3,  0.5,  0.1]], dtype=torch.float64)
b_mu = torch.tensor([0.1, -0.1], dtype=torch.float64)

W_v  = torch.tensor([[ 0.2,  0.3, -0.4],
                     [ 0.1, -0.2,  0.3]], dtype=torch.float64)
b_v  = torch.tensor([0.0, 0.0], dtype=torch.float64)

x = torch.tensor([1.0, 0.5, -1.0], dtype=torch.float64)

# ── Compute mu and log_var ────────────────────────────────────────────────────
mu      = W_mu @ x + b_mu
log_var = W_v  @ x + b_v
sigma2  = torch.exp(log_var)
sigma   = torch.sqrt(sigma2)

print('mu:        ', mu.numpy().round(4))
print('log_var:   ', log_var.numpy().round(4))
print('sigma^2:   ', sigma2.numpy().round(4))
print('sigma:     ', sigma.numpy().round(4))

mu:         [-0.2  -0.25]
log_var:    [ 0.75 -0.3 ]
sigma^2:    [2.117  0.7408]
sigma:      [1.455  0.8607]


Expected from Note 30:
- `mu` = [-0.20, -0.25]
- `log_var` = [0.75, -0.30]
- `sigma^2` ≈ [2.1170, 0.7408]
- `sigma` ≈ [1.4550, 0.8607]

## Part 2 — Reparameterisation and KL Divergence

We apply the reparameterisation trick with the fixed noise sample from the worked example, then compute the closed-form KL divergence.

In [2]:
# ── Reparameterisation: z = mu + sigma * eps ──────────────────────────────────
eps = torch.tensor([0.5, -1.0], dtype=torch.float64)
z   = mu + sigma * eps
print('z (reparameterised): ', z.numpy().round(4))

# ── KL divergence (closed form) ───────────────────────────────────────────────
# KL_j = 0.5 * (sigma^2_j + mu^2_j - 1 - log_var_j)
kl_per_dim = 0.5 * (sigma2 + mu**2 - 1 - log_var)
kl_total   = kl_per_dim.sum()

print('KL per dimension:    ', kl_per_dim.numpy().round(4))
print('KL total:            ', round(kl_total.item(), 4))

z (reparameterised):  [ 0.5275 -1.1107]
KL per dimension:     [0.2035 0.0517]
KL total:             0.2552


Expected from Note 30:
- `z` ≈ [0.5275, -1.1107]
- `KL per dimension` ≈ [0.2035, 0.0517]
- `KL total` ≈ 0.2552

## Part 3 — Full VAE on Synthetic Data

We implement a complete VAE with PyTorch's `nn.Module`. The encoder outputs $\mu$ and $\log\sigma^2$; the reparameterisation is applied inside the forward pass; the loss is reconstruction MSE plus KL divergence.

In [3]:
torch.manual_seed(0)

class VAE(nn.Module):
    def __init__(self, d, k):
        super().__init__()
        self.enc_shared = nn.Sequential(nn.Linear(d, 16), nn.Tanh())
        self.fc_mu      = nn.Linear(16, k)
        self.fc_logvar  = nn.Linear(16, k)
        self.decoder    = nn.Sequential(nn.Linear(k, 16), nn.Tanh(), nn.Linear(16, d))

    def encode(self, x):
        h = self.enc_shared(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterise(self, mu, log_var):
        sigma = torch.exp(0.5 * log_var)
        eps   = torch.randn_like(sigma)
        return mu + sigma * eps

    def forward(self, x):
        mu, log_var = self.encode(x)
        z           = self.reparameterise(mu, log_var)
        x_hat       = self.decoder(z)
        return x_hat, mu, log_var


def vae_loss(x, x_hat, mu, log_var):
    recon = ((x - x_hat) ** 2).mean()
    kl    = (-0.5 * (1 + log_var - mu**2 - torch.exp(log_var))).mean()
    return recon + kl, recon, kl


# ── Synthetic data: 300 samples, d=8 ─────────────────────────────────────────
X = torch.randn(300, 8)

vae       = VAE(d=8, k=2)
optimiser = torch.optim.Adam(vae.parameters(), lr=1e-3)

for epoch in range(500):
    x_hat, mu_b, lv_b = vae(X)
    loss, recon, kl   = vae_loss(X, x_hat, mu_b, lv_b)
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()

print(f'Final total loss:        {loss.item():.4f}')
print(f'Final reconstruction:    {recon.item():.4f}')
print(f'Final KL:                {kl.item():.4f}')

Final total loss:        1.0160
Final reconstruction:    1.0156
Final KL:                0.0004


## Part 4 — Generation: Sampling from the Prior

After training, we generate new samples by drawing $z \sim \mathcal{N}(0, I)$ and passing through the decoder. We also check that the approximate posterior mean is close to zero — confirming the KL term has regularised the latent space.

In [4]:
vae.eval()
with torch.no_grad():
    # ── Generate 5 new samples ────────────────────────────────────────────────
    z_samples  = torch.randn(5, 2)
    x_generated = vae.decoder(z_samples)
    print('Generated samples (5 x 8):')
    print(x_generated.numpy().round(3))

    # ── Check posterior means are near zero ───────────────────────────────────
    _, mu_all, lv_all = vae(X)
    print(f'\nMean of posterior mu (should be near 0): {mu_all.mean().item():.4f}')
    print(f'Mean of posterior sigma^2 (should be near 1): {torch.exp(lv_all).mean().item():.4f}')

Generated samples (5 x 8):
[[-0.102  0.132 -0.073  0.086  0.088 -0.063  0.129 -0.104]
 [-0.123  0.101 -0.006 -0.007  0.069  0.027  0.087 -0.081]
 [-0.15   0.124 -0.013 -0.035  0.053  0.045  0.07  -0.062]
 [-0.168  0.107 -0.019  0.055  0.033  0.038  0.028 -0.118]
 [-0.096  0.105 -0.039  0.073  0.075 -0.     0.08  -0.096]]

Mean of posterior mu (should be near 0): 0.0004
Mean of posterior sigma^2 (should be near 1): 0.9975


The posterior mean should be close to zero and the posterior variance close to one, confirming the KL term has pushed the approximate posterior towards the standard Gaussian prior.

## Summary

This notebook demonstrated:
1. Encoder forward pass — computing $\mu$, $\log\sigma^2$, $\sigma$, matching the Note 30 worked example exactly
2. Reparameterisation trick — $z = \mu + \sigma \odot \varepsilon$, and closed-form KL divergence
3. Full VAE training with ELBO loss (reconstruction MSE + KL divergence)
4. Generation by sampling $z \sim \mathcal{N}(0, I)$ and decoding; posterior statistics confirm latent space regularisation